In [4]:
# Parametrar
pensionsalder = 65
franta = 1.016
year_filter = 0               #           0 => no filter

In [5]:
# Vi importerar det aggregerade datasetet från Morrins studie
import pandas as pd
from math import floor
df = pd.read_csv(r".\File_9_LifeExpectancy_DecilesIncome_IndividualIncome.csv")

# Filtrerar
if year_filter > 0:
    df = df[df['year'] == year_filter]

In [6]:
# Fristående lönesimulering, ett 10 000 lognormala löner har simulerats med en fördelning peggad mot siffror från pensionsmyndigheten, sedan har fördelningen
# delats in i 10 lika stor grupper, där gruppmedel anges nedan
kvinnor_v0 = {  "1":780997,
                "2":1125171,
                "3":1392136,
                "4":1650803,
                "5":1921606,
                "6":2244620,
                "7":2635908,
                "8":3132064,
                "9":3865826,
                "10":5502823}

man_v0 = {  "1":875172,
            "2":1260112,
            "3":1570483,
            "4":1888806,
            "5":2213281,
            "6":2595123,
            "7":3071718,
            "8":3685956,
            "9":4655568,
            "10":6855691}

df_temp_k = pd.DataFrame([kvinnor_v0.keys(), kvinnor_v0.values()]).T
df_temp_k["sex"] = 2

df_temp_m = pd.DataFrame([man_v0.keys(), man_v0.values()]).T
df_temp_m["sex"] = 1

df_lon = pd.concat([df_temp_k, df_temp_m]).reset_index().iloc[:, 1:]
df_lon.columns = ["level_income", "v0", "sex"]
df_lon["level_income"] = df_lon["level_income"].astype("Int64")


In [7]:
def berakna_delningstal(df, income_filter, sex_filter, franta, pensionsalder):
    df_filtered = df.copy()
    i = pensionsalder

    if income_filter > 0:
        df_filtered = df_filtered.copy()[df_filtered["level_income"] == income_filter]
    if sex_filter > 0:
        df_filtered = df_filtered.copy()[df_filtered["sex"] == sex_filter]

    temp = df_filtered.groupby(["age"]).mean().reset_index()[["age", "_lx"]]

    ages = min(len(temp[~temp["_lx"].isna()]), 45)
    L = {temp["age"][j] : temp["_lx"][j] for j in range(ages)}

    scaler = 1 / (12 * L[i])

    Di = 0

    for k in range (i, i + ages -5):
        for X in range(0, 12):
            val1 = (L[k] + (L[k+1] - L[k]) * X / 12)
            val2 = franta**-(k-i) * franta**-(X/12)
            Di += scaler * val1 * val2


    return Di 


In [8]:
# Tabell för att demonstrera beståndsstatistiken
df_table1 = df[(df['age'] == 65) & (df['sex'] == 1)].groupby(["sex", "level_income", "year"]).mean().reset_index()[["sex", "level_income", "year", "_ExpYL"]].copy(deep=True)
df_table1 = df_table1[df_table1['year'].isin([2006, 2008, 2010, 2012, 2014])]
df_table1 = df_table1.pivot(index = ['sex', 'level_income'], columns='year').reset_index()
#df_table1


In [9]:
# Vi grupperar enligt våra 20 typfall och joinar in pensionsbehållningen
df_sex_income_yl = df[df['age'] == pensionsalder].groupby(["sex", "level_income"]).mean().reset_index()[["sex", "level_income", "_ExpYL"]].copy(deep=True)
df_sex_income_yl["overall_mean"] = df_sex_income_yl["_ExpYL"].mean()
df_final = df_sex_income_yl.merge(df_lon, how="left", on = ["sex", "level_income"])
helper = lambda a,b,c : berakna_delningstal(df, a, b, c, pensionsalder)

# Bestämmer faktisk återstående livslängd med förskottsränta 0 % (alltså franta = 1.0)
df_final['ExpYL'] = df_final.apply(lambda x: helper(x.level_income, x.sex, 1), axis=1)

# Bestämmer delningstal enligt dagens formel
df_final['delningstal_dagens'] = df_final.apply(lambda x: helper(0, 0, franta), axis=1)

# Bestämmer delningstal enligt ny formel där pengars livslängd korrelerar med beloppet
df_final['delningstal'] = df_final['delningstal_dagens'] + (df_final['v0']) / 1000000

# Normaliserar så totala antalet utbetalningsår är intakt
df_final['delningstal'] *= (df_final['delningstal_dagens'].sum() / df_final['delningstal'].sum())
df_final['startbelopp'] = df_final['v0'] / df_final['delningstal'] / 12

df_final['total_utbetalt_2.4'] = 12 * df_final['startbelopp'] * ((1 + 1.024 - franta)**( df_final['ExpYL'].apply(floor)) - 1) / (1.024 - franta) + (df_final['ExpYL'] - df_final['ExpYL'].apply(floor))*(1 + 1.024 - franta)**df_final['ExpYL']
df_final['total_utbetalt_0.8'] = 12 * df_final['startbelopp'] * ((1 + 1.008 - franta)**(df_final['ExpYL'].apply(floor)) - 1) / (1.008 - franta) + (df_final['ExpYL'] - df_final['ExpYL'].apply(floor))*(1 + 1.008 - franta)**df_final['ExpYL']



df_final['scenario_2.4_erhallet'] = df_final['total_utbetalt_2.4'] / df_final['v0']
df_final['scenario_1.6_erhallet'] = df_final['ExpYL'] / df_final['delningstal']
df_final['scenario_0.8_erhallet'] = df_final['total_utbetalt_0.8'] / df_final['v0']
df_final.head(100)

# Vi antar konstant utveckling av löneindex, 0.8, 1.6 eller 2.4 eller 3.2
# Totalt utbetalt belopp bör vara oförändrat, kan vi normalisera och täta på något sätt? 


,sex,level_income,_ExpYL,overall_mean,v0,ExpYL,delningstal_dagens,delningstal,startbelopp,total_utbetalt_2.4,total_utbetalt_0.8,scenario_2.4_erhallet,scenario_1.6_erhallet,scenario_0.8_erhallet
0,1,1,16.380604,19.860024,875172,16.428258,16.687392,15.158771,4811.142046,981288.140171,870331.687141,1.121252,1.083746,0.994469
1,1,2,16.637789,19.860024,1260112,16.686364,16.687392,15.491024,6778.721215,1382598.795141,1226265.231559,1.097203,1.077163,0.97314
2,1,3,17.542553,19.860024,1570483,17.589449,16.687392,15.758914,8304.733394,1807053.958266,1589957.637117,1.150636,1.116159,1.0124
3,1,4,17.986077,19.860024,1888806,18.031434,16.687392,16.033669,9816.873773,2270975.900631,1982226.137217,1.202334,1.124598,1.04946
4,1,5,18.273008,19.860024,2213281,18.313483,16.687392,16.313733,11305.817532,2615419.419165,2282874.352935,1.181693,1.122581,1.031444
5,1,6,18.657151,19.860024,2595123,18.699774,16.687392,16.643312,12993.823205,3005912.833391,2623717.298765,1.158293,1.123561,1.011018
6,1,7,19.032392,19.860024,3071718,19.062787,16.687392,17.054675,15009.169017,3680017.002517,3186520.764787,1.198032,1.117746,1.037374
7,1,8,19.577946,19.860024,3685956,19.609655,16.687392,17.584842,17467.486957,4282759.314543,3708434.271558,1.161913,1.115145,1.006098
8,1,9,20.222358,19.860024,4655568,20.264523,16.687392,18.421743,21060.113546,5457645.869346,4688118.377578,1.172284,1.100033,1.006992
9,1,10,21.104795,19.860024,6855691,21.141838,16.687392,20.320735,28114.514314,7681426.225876,6545780.115626,1.120445,1.040407,0.954795
